# XGBoost: pipeline training and 2019 evaluation
This notebook calls the same `XGBoostModel` trainer used by the pipeline in
`Models/xgboost_model.py` and reproduces the 2019 split and evaluation filter in
`Run Pipeline/additional_funcs/automated_model_selection.py`.

Training pools all elections before 2019 and retains every winner class, including
`oth`. Rows without winners are dropped by the trainer. Within each of five
stratified cross-validation folds, categorical predictors are one-hot encoded
with unknown categories ignored; numeric predictors pass through to XGBoost,
which handles their missing values. The accuracy grid searches 20, 35, 50, and
100 estimators with depths 2, 3, 4, and 5, then refits the best configuration on
all labelled pre-2019 rows. The trainer sets the XGBoost random seed to 42.

Evaluation uses only complete 2019 rows, exactly as in model selection. Changed
seats are those whose winner differs from a known previous winner. The notebook
stops after this evaluation, before the pipeline's selected-model retraining.


In [9]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.metrics import accuracy_score

project_root = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "TEST_TRAIN" / "train.csv").is_file()), None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the election project.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Models.xgboost_model import XGBoostModel
from Models.logistic_regression import FEATURE_COLUMNS as VALIDATION_FEATURE_COLUMNS

# The model-selection pipeline uses logistic regression's feature list to
# give every candidate the same complete validation rows.
data = pd.read_csv(project_root / "TEST_TRAIN" / "train.csv")
election_years = pd.to_numeric(data["election"], errors="raise")
if election_years.isna().any():
    raise ValueError("Election years must not be missing.")
training_data = data.loc[election_years < 2019].copy()
validation_data = data.loc[election_years == 2019].dropna(
    subset=VALIDATION_FEATURE_COLUMNS + ["winner"]
)
if training_data.empty:
    raise ValueError("Data must contain training rows from elections before 2019.")
if validation_data.empty:
    raise ValueError("Data must contain complete validation rows for the 2019 election.")
print(f"Training: {len(training_data)} rows before 2019")
print(f"Evaluation: {len(validation_data)} complete 2019 rows")


Training: 5073 rows before 2019
Evaluation: 629 complete 2019 rows


In [10]:
model = XGBoostModel()
model.train(training_data)
classifier = model.pipeline.named_steps["classifier"]
print(f"Selected n_estimators={classifier.n_estimators}, max_depth={classifier.max_depth}")


Selected n_estimators=20, max_depth=2


In [11]:
predictions = model.predict(validation_data)
accuracy = float(accuracy_score(validation_data["winner"], predictions))
changed_seats = (
    validation_data["previous_winner"].notna()
    & validation_data["winner"].ne(validation_data["previous_winner"])
).to_numpy(dtype=bool)
changed_seat_count = int(changed_seats.sum())
changed_accuracy = (
    float(accuracy_score(
        validation_data.loc[changed_seats, "winner"],
        predictions[changed_seats],
    ))
    if changed_seat_count else None
)
selection_score = accuracy + (changed_accuracy if changed_accuracy is not None else 0.0)
changed_display = f"{changed_accuracy:.2%}" if changed_accuracy is not None else "N/A"
print(f"XGBoost 2019 accuracy: {accuracy:.2%} ({len(validation_data)} complete seats)")
print(f"2019 changed-seat accuracy: {changed_display} ({changed_seat_count} seats)")
print(f"Pipeline combined selection score: {selection_score:.4f}")
results = pd.DataFrame([{
    "model": model.name,
    "evaluation_year": 2019,
    "evaluation_rows": len(validation_data),
    "accuracy": accuracy,
    "changed_seat_evaluation_rows": changed_seat_count,
    "changed_seat_accuracy": changed_accuracy,
    "selection_score": selection_score,
    "n_estimators": classifier.n_estimators,
    "max_depth": classifier.max_depth,
}])
results


XGBoost 2019 accuracy: 87.76% (629 complete seats)
2019 changed-seat accuracy: 4.00% (75 seats)
Pipeline combined selection score: 0.9176


,model,evaluation_year,evaluation_rows,accuracy,changed_seat_evaluation_rows,changed_seat_accuracy,selection_score,n_estimators,max_depth
0,XGBoost,2019,629,0.877583,75,0.04,0.917583,20,2


## Brier score and log loss (cross-entropy)
The multiclass Brier score is the mean, across seats, of the sum over all parties
of squared differences between predicted probabilities and one-hot outcomes.
This uses the unscaled multiclass convention (range 0–2).
Log loss and cross-entropy are the same metric here: the mean negative natural
logarithm of the actual winner's probability. Lower is better for both metrics.
An unseen winning party receives probability zero; log loss clips probabilities
to float64 machine epsilon to avoid infinite values. Metrics use the same
2019 evaluation subsets as the accuracy results above.


In [12]:
import numpy as np


def probability_scores(winners, probabilities, classes):
    """Mean multiclass Brier score and cross-entropy, with natural logarithms."""
    # Include unseen winning parties with predicted probability zero.
    labels = list(classes)
    labels += [label for label in pd.unique(np.asarray(winners)) if label not in labels]
    probability_table = pd.DataFrame(
        np.asarray(probabilities, dtype=float), columns=classes
    ).reindex(columns=labels, fill_value=0.0)
    probabilities = probability_table.to_numpy()
    winner_columns = pd.Index(labels).get_indexer(np.asarray(winners))
    targets = np.eye(len(labels))[winner_columns]
    brier = float(np.mean(np.sum((probabilities - targets) ** 2, axis=1)))
    winning_probabilities = probabilities[np.arange(len(winner_columns)), winner_columns]
    # Clip zero probabilities to keep log loss finite, including unseen winners.
    log_loss = float(-np.mean(np.log(np.clip(
        winning_probabilities, np.finfo(float).eps, 1.0
    ))))
    return {"brier_score": brier, "log_loss_cross_entropy": log_loss}

probability_metrics = probability_scores(
    validation_data["winner"],
    model.pipeline.predict_proba(validation_data),
    model.pipeline.classes_,
)
print(f"2019 probability metrics ({len(validation_data)} complete seats):")
print(f"Brier score: {probability_metrics['brier_score']:.6f}")
print(f"Log loss (cross-entropy): {probability_metrics['log_loss_cross_entropy']:.6f}")
for metric, value in probability_metrics.items():
    results[metric] = value
results


2019 probability metrics (629 complete seats):
Brier score: 0.189248
Log loss (cross-entropy): 0.318404


,model,evaluation_year,evaluation_rows,accuracy,changed_seat_evaluation_rows,changed_seat_accuracy,selection_score,n_estimators,max_depth,brier_score,log_loss_cross_entropy
0,XGBoost,2019,629,0.877583,75,0.04,0.917583,20,2,0.189248,0.318404
